# RelCheck — COCO Image Screening for RelTR

Finds COCO val2014 images that contain entity pairs RelTR can detect.
RelTR is trained on Visual Genome (VG-150 objects, VG-50 predicates).
Images with multiple VG-150-mapped entities will produce richer scene graphs.

**Output:** `reltr_friendly_images.json` — list of image paths ranked by RelTR coverage score.

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
import os, json, glob, zipfile, shutil
from collections import defaultdict
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT    = "/content/drive/MyDrive/RelCheck_Data"
COCO_ZIP_DIR   = f"{GDRIVE_ROOT}/coco_zips"
COCO_IMG_DIR   = f"{COCO_ZIP_DIR}/val2014"          # images downloaded on-demand in Cell 4
COCO_ANN_PATH  = f"{COCO_ZIP_DIR}/annotations/instances_val2014.json"
OUTPUT_PATH    = f"{GDRIVE_ROOT}/reltr_friendly_images.json"

N_SELECT = 100   # how many images to select

os.makedirs(COCO_IMG_DIR, exist_ok=True)

# ── Auto-extract annotations zip if needed ────────────────────────────────────
if not os.path.exists(COCO_ANN_PATH):
    ann_zip = f"{COCO_ZIP_DIR}/annotations_trainval2014.zip"
    if os.path.exists(ann_zip):
        print(f"Extracting annotations from {os.path.basename(ann_zip)} ...")
        with zipfile.ZipFile(ann_zip, 'r') as z:
            z.extract("annotations/instances_val2014.json", COCO_ZIP_DIR)
        print(f"  ✅ {COCO_ANN_PATH}  ({os.path.getsize(COCO_ANN_PATH)/1e6:.1f} MB)")
    else:
        print(f"❌ Annotations not found. Upload one of:")
        print(f"   {COCO_ANN_PATH}")
        print(f"   {ann_zip}")
else:
    print(f"✅ Annotations ready  ({os.path.getsize(COCO_ANN_PATH)/1e6:.1f} MB)")

print(f"Output: {OUTPUT_PATH}")
print(f"\nNOTE: Images are downloaded on-demand in Cell 4 — no need to upload val2014.zip")


In [ ]:
# ── CELL 2: Define good entity pairs for RelTR ────────────────────────────────
#
# These are COCO category names → VG-150 class names.
# RelTR was trained on VG so these pairs produce reliable scene graph triples.
# Pairs are scored by expected relational richness (how often VG has a clear
# predicate between them beyond just 'near' or 'and').

# COCO category name → VG-150 class name (or None if no good mapping)
COCO_TO_VG = {
    'person':       'person',
    'bicycle':      'bike',
    'motorcycle':   'motorcycle',
    'horse':        'horse',
    'dog':          'dog',
    'cat':          'cat',
    'bird':         'bird',
    'elephant':     'elephant',
    'bear':         'bear',
    'giraffe':      'giraffe',
    'cow':          'cow',
    'sheep':        'sheep',
    'skateboard':   'skateboard',
    'surfboard':    'surfboard',
    'skis':         'ski',
    'snowboard':    'ski',
    'kite':         'kite',
    'umbrella':     'umbrella',
    'chair':        'chair',
    'couch':        'chair',
    'dining table': 'table',
    'car':          'car',
    'truck':        'truck',
    'bus':          'bus',
    'train':        'train',
    'boat':         'boat',
    'airplane':     'airplane',
    'backpack':     'bag',
    'handbag':      'bag',
    'tie':          'tie',
    'hat':          'hat',       # not a COCO class but often detected
    'laptop':       'laptop',
    'cell phone':   'phone',
    'book':         'book',
    'clock':        'clock',
    'vase':         'vase',
    'bottle':       'bottle',
    'cup':          'cup',
    'bowl':         'bowl',
    'pizza':        'pizza',
    'banana':       'banana',
    'orange':       'orange',
    'tennis racket':'racket',
}

# High-value entity pairs: RelTR reliably predicts non-trivial predicates
# (not just 'near', 'and', 'of') for these combinations
# Score = expected RelTR usefulness (3=high, 2=medium, 1=low)
HIGH_VALUE_PAIRS = [
    # (coco_cat_1, coco_cat_2, score, typical_vg_predicates)
    ('person', 'horse',        3, 'riding, holding, near'),
    ('person', 'bicycle',      3, 'riding, on, next to'),
    ('person', 'motorcycle',   3, 'riding, on, sitting on'),
    ('person', 'skateboard',   3, 'riding, on, standing on'),
    ('person', 'surfboard',    3, 'riding, carrying, on'),
    ('person', 'skis',         3, 'wearing, on, using'),
    ('person', 'snowboard',    3, 'riding, on, standing on'),
    ('person', 'elephant',     3, 'riding, standing near, feeding'),
    ('person', 'kite',         2, 'flying, holding, carrying'),
    ('person', 'umbrella',     2, 'holding, carrying, under'),
    ('person', 'dog',          2, 'walking, holding, with'),
    ('person', 'giraffe',      2, 'feeding, near, looking at'),
    ('person', 'bear',         2, 'near, looking at'),
    ('cat',    'chair',        3, 'sitting on, on, lying on'),
    ('cat',    'couch',        3, 'sitting on, on, lying on'),
    ('dog',    'couch',        3, 'sitting on, lying on, on'),
    ('dog',    'chair',        2, 'sitting on, on'),
    ('person', 'chair',        2, 'sitting on, standing near'),
    ('person', 'dining table', 2, 'sitting at, standing at'),
    ('bird',   'person',       2, 'flying over, near, on'),
    ('cow',    'person',       1, 'near, behind'),
    ('person', 'tennis racket',2, 'holding, using, carrying'),
    ('person', 'laptop',       2, 'using, holding, looking at'),
    ('person', 'cell phone',   2, 'holding, using, talking on'),
    ('person', 'backpack',     2, 'wearing, carrying'),
    ('person', 'handbag',      2, 'carrying, holding'),
]

# Build lookup: coco_cat → set of coco_cats it pairs well with
pair_score = {}
for c1, c2, score, _ in HIGH_VALUE_PAIRS:
    pair_score[(c1, c2)] = score
    pair_score[(c2, c1)] = score

print(f"{len(COCO_TO_VG)} COCO categories mapped to VG-150")
print(f"{len(HIGH_VALUE_PAIRS)} high-value entity pairs defined")

In [ ]:
# ── CELL 3: Load COCO annotations + score images ──────────────────────────────
print("Loading COCO annotations...")
with open(COCO_ANN_PATH) as f:
    coco = json.load(f)

# Build category id → name lookup
cat_id_to_name = {c['id']: c['name'] for c in coco['categories']}
print(f"  {len(cat_id_to_name)} COCO categories")
print(f"  {len(coco['images'])} total images")
print(f"  {len(coco['annotations'])} annotations")

# Build image_id → set of category names
img_categories = defaultdict(set)
for ann in coco['annotations']:
    cat_name = cat_id_to_name.get(ann['category_id'], '')
    img_categories[ann['image_id']].add(cat_name)

# Build image_id → file info
img_info = {img['id']: img for img in coco['images']}

# Score each image
scored_images = []
for img_id, cats in img_categories.items():
    cats_list = list(cats)

    # Check how many high-value pairs are present
    total_score = 0
    matched_pairs = []
    vg_mapped_cats = [c for c in cats_list if c in COCO_TO_VG]

    for i, c1 in enumerate(cats_list):
        for c2 in cats_list[i+1:]:
            s = pair_score.get((c1, c2), 0)
            if s > 0:
                total_score += s
                matched_pairs.append((c1, c2, s))

    if total_score == 0:
        continue

    info = img_info.get(img_id, {})
    fname = info.get('file_name', f"{img_id:012d}.jpg")
    fpath = os.path.join(COCO_IMG_DIR, fname)

    scored_images.append({
        'image_id':      img_id,
        'file_name':     fname,
        'file_path':     fpath,
        'score':         total_score,
        'categories':    cats_list,
        'vg_cats':       vg_mapped_cats,
        'matched_pairs': matched_pairs,
    })

# Sort by score descending
scored_images.sort(key=lambda x: -x['score'])

print(f"\n{len(scored_images)} images have at least one high-value entity pair")
print(f"Score distribution:")
from collections import Counter
score_dist = Counter(img['score'] for img in scored_images)
for score in sorted(score_dist.keys(), reverse=True)[:8]:
    print(f"  score={score}: {score_dist[score]} images")

In [ ]:
# ── CELL 4: Select top N images + download only those ─────────────────────────
# Scores were computed from annotations only (no images needed).
# Now select the top N_SELECT images with diversity, then download
# just those specific files from the COCO image server (~50-100 MB total).

import requests as _req
from collections import defaultdict

COCO_IMG_URL = "http://images.cocodataset.org/val2014"

# ── Diversity-constrained selection from scored list ──────────────────────────
selected = []
pair_type_counts = defaultdict(int)
MAX_PER_PAIR_TYPE = max(5, N_SELECT // len(HIGH_VALUE_PAIRS))

for img in scored_images:
    if len(selected) >= N_SELECT:
        break
    if img['matched_pairs']:
        best_pair = max(img['matched_pairs'], key=lambda x: x[2])
        pair_key  = tuple(sorted([best_pair[0], best_pair[1]]))
        if pair_type_counts[pair_key] < MAX_PER_PAIR_TYPE:
            pair_type_counts[pair_key] += 1
            selected.append(img)

# Fill remainder without diversity constraint if short
if len(selected) < N_SELECT:
    selected_ids = {img['image_id'] for img in selected}
    for img in scored_images:
        if len(selected) >= N_SELECT: break
        if img['image_id'] not in selected_ids:
            selected.append(img)

print(f"Selected {len(selected)} images (target={N_SELECT})")
print("\nPair type breakdown:")
for pair, count in sorted(pair_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {pair[0]} + {pair[1]}: {count}")

# ── Download only the selected images ─────────────────────────────────────────
print(f"\nDownloading {len(selected)} images to {COCO_IMG_DIR} ...")
already, downloaded, failed = 0, 0, 0

for i, img in enumerate(selected):
    fname = img['file_name']   # e.g. COCO_val2014_000000123456.jpg
    fpath = os.path.join(COCO_IMG_DIR, fname)
    img['file_path'] = fpath   # update to local path

    if os.path.exists(fpath):
        already += 1
        continue

    url = f"{COCO_IMG_URL}/{fname}"
    try:
        r = _req.get(url, timeout=15)
        if r.status_code == 200:
            with open(fpath, 'wb') as f:
                f.write(r.content)
            downloaded += 1
        else:
            print(f"  ✗ {fname}  HTTP {r.status_code}")
            failed += 1
    except Exception as e:
        print(f"  ✗ {fname}  {e}")
        failed += 1

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(selected)}  "
              f"downloaded={downloaded}  cached={already}  failed={failed}")

print(f"\nDone: {downloaded} new, {already} cached, {failed} failed")
print(f"Total on disk: {len(os.listdir(COCO_IMG_DIR))} images")


In [ ]:
# ── CELL 5: Preview + save ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image

# Show top 6 images
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_info in zip(axes.flat, selected[:6]):
    try:
        im = Image.open(img_info['file_path']).convert('RGB')
        ax.imshow(im)
        pairs_str = ', '.join(f"{p[0]}+{p[1]}" for p in img_info['matched_pairs'][:2])
        ax.set_title(f"score={img_info['score']}\n{pairs_str}", fontsize=9)
    except Exception as e:
        ax.set_title(f"Load error: {e}")
    ax.axis('off')
plt.suptitle('Top selected images (RelTR-friendly entity pairs)', fontsize=12)
plt.tight_layout()
plt.savefig(f"{GDRIVE_ROOT}/reltr_selected_preview.png", dpi=100)
plt.show()

# Save output
output = {
    'n_selected':      len(selected),
    'n_total_scored':  len(scored_images),
    'images': [
        {
            'image_id':      img['image_id'],
            'file_path':     img['file_path'],
            'file_name':     img['file_name'],
            'score':         img['score'],
            'vg_categories': img['vg_cats'],
            'matched_pairs': img['matched_pairs'],
        }
        for img in selected
    ]
}

with open(OUTPUT_PATH, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Saved {len(selected)} image paths → {OUTPUT_PATH}")
print("\nTop 10 images by score:")
print(f"{'Image ID':<15} {'Score':>5}  {'Pairs'}")
print('-' * 60)
for img in selected[:10]:
    pairs = ', '.join(f"{p[0]}+{p[1]}(s={p[2]})" for p in img['matched_pairs'])
    print(f"{img['image_id']:<15} {img['score']:>5}  {pairs}")


In [ ]:
# ── CELL 6: Verify RelTR actually fires on a sample ───────────────────────────
# Loads RelTR and runs it on the top 3 selected images.
# ★ = non-trivial predicate (riding/sitting on/wearing etc.) — good for correction
#   = trivial (near/and/of) — not useful

import sys, argparse, torch
import torchvision.transforms as T
from PIL import Image

# ── Clone RelTR + create ckpt folder ──────────────────────────────────────────
if not os.path.exists('RelTR'):
    !git clone -q https://github.com/yrcong/RelTR.git
    print("RelTR cloned ✅")

os.makedirs('RelTR/ckpt', exist_ok=True)

RELTR_CKPT = 'RelTR/ckpt/checkpoint0149.pth'
if not os.path.exists(RELTR_CKPT):
    print(f"⚠️  Checkpoint not found at {RELTR_CKPT}")
    print(f"   Upload checkpoint0149.pth via the Colab Files panel:")
    print(f"   Files → RelTR → ckpt → upload checkpoint0149.pth")
    print(f"   Download from: https://github.com/yrcong/RelTR (README link)")
    raise FileNotFoundError("Upload checkpoint0149.pth to RelTR/ckpt/ first")

# ── Load model (skip if already loaded) ───────────────────────────────────────
sys.path.insert(0, 'RelTR')

if 'reltr_model' not in dir() or reltr_model is None:
    from models import build_model as reltr_build_model

    RELTR_CLASSES = [
        'N/A','airplane','animal','arm','bag','banana','basket','beach','bear',
        'bed','bench','bike','bird','board','boat','book','boot','bottle','bowl',
        'box','boy','branch','building','bus','cabinet','cap','car','cat','chair',
        'child','clock','coat','counter','cow','cup','curtain','desk','dog','door',
        'drawer','ear','elephant','engine','eye','face','fence','finger','flag',
        'flower','food','fork','fruit','giraffe','girl','glass','glove','guy',
        'hair','hand','handle','hat','head','helmet','hill','horse','house',
        'jacket','jean','kid','kite','lady','lamp','laptop','leaf','leg','letter',
        'light','logo','man','men','motorcycle','mountain','mouth','neck','nose',
        'number','orange','pant','paper','paw','people','person','phone','pillow',
        'pizza','plane','plant','plate','player','pole','post','pot','racket',
        'railing','rock','roof','room','screen','seat','sheep','shelf','shirt',
        'shoe','short','sidewalk','sign','sink','skateboard','ski','skier',
        'sleeve','snow','sock','stand','street','surfboard','table','tail','tie',
        'tile','tire','toilet','towel','tower','track','train','tree','truck',
        'trunk','umbrella','vase','vegetable','vehicle','wave','wheel','window',
        'windshield','wing','wire','woman','wood'
    ]
    RELTR_REL_CLASSES = [
        'N/A','above','across','against','along','and','at','attached to',
        'behind','belonging to','between','carrying','covered in','covering',
        'eating','flying in','for','from','growing on','hanging from','has',
        'holding','in','in front of','laying on','looking at','lying on',
        'made of','mounted on','near','of','on','on back of','over','painted on',
        'parked on','part of','playing','riding','says','sitting on','standing on',
        'to','under','using','walking in','walking on','watching','wearing','wears','with'
    ]

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    reltr_args = argparse.Namespace(
        backbone='resnet50', dilation=False, lr_backbone=0,
        return_interm_layers=False, position_embedding='sine',
        enc_layers=6, dec_layers=6, dim_feedforward=2048,
        hidden_dim=256, dropout=0.1, nheads=8,
        num_entities=100, num_triplets=200, pre_norm=False,
        set_cost_class=1, set_cost_bbox=5, set_cost_giou=2,
        set_iou_threshold=0.7, bbox_loss_coef=5, giou_loss_coef=2,
        rel_loss_coef=1, eos_coef=0.1, aux_loss=False,
        dataset='vg', device=device,
    )
    reltr_model, _, _ = reltr_build_model(reltr_args)
    ckpt = torch.load(RELTR_CKPT, map_location=device, weights_only=False)
    reltr_model.load_state_dict(ckpt['model'])
    reltr_model.eval().to(device)
    print(f"RelTR loaded ✅  ({os.path.getsize(RELTR_CKPT)/1e6:.0f} MB)  device={device}")

# Always define transform (needed whether model was just loaded or cached)
reltr_transform = T.Compose([
    T.Resize(800), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Spot-check top 3 images ───────────────────────────────────────────────────
CONF_THRESHOLD = 0.3
NON_TRIVIAL = {'riding','sitting on','standing on','lying on','laying on',
               'wearing','wears','holding','carrying','eating','mounted on',
               'parked on','walking on','on back of','attached to'}

print("\nRelTR spot-check on top 3 selected images:")
for img_data in selected[:3]:
    img = Image.open(img_data['file_path']).convert('RGB')
    img_t = reltr_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = reltr_model(img_t)

    probas     = outputs['rel_logits'].softmax(-1)[0, :, :-1]
    probas_sub = outputs['sub_logits'].softmax(-1)[0, :, :-1]
    probas_obj = outputs['obj_logits'].softmax(-1)[0, :, :-1]
    keep = torch.nonzero(
        (probas.max(-1)[0] > CONF_THRESHOLD) &
        (probas_sub.max(-1)[0] > CONF_THRESHOLD) &
        (probas_obj.max(-1)[0] > CONF_THRESHOLD)
    ).flatten()

    triples = sorted([
        (RELTR_CLASSES[probas_sub[i].argmax()],
         RELTR_REL_CLASSES[probas[i].argmax()],
         RELTR_CLASSES[probas_obj[i].argmax()],
         probas[i].max().item())
        for i in keep
    ], key=lambda x: -x[3])

    non_trivial = [t for t in triples if t[1] in NON_TRIVIAL]
    pairs_str   = ', '.join(f"{p[0]}+{p[1]}" for p in img_data['matched_pairs'])
    print(f"\n  [{img_data['image_id']}]  COCO pairs: {pairs_str}")
    print(f"  {len(triples)} triples total, {len(non_trivial)} non-trivial")
    for s, p, o, c in triples[:6]:
        flag = '★' if p in NON_TRIVIAL else ' '
        print(f"    {flag} ({s}, {p}, {o})  conf={c:.2f}")

print("\n★ = non-trivial predicate (good for correction)")
